# 01 Bronze: raw CSV files to DuckDB (incremental)

This notebook loads the CSV files from `data/raw` into the `bronze` schema of `data/crossselliq.duckdb`.

Bronze rules:

- The content stays unchanged: all columns, all values stored as text (`VARCHAR`). Types and cleaning belong to silver.
- Every row gets two metadata columns: `_source_file` (which CSV it came from) and `_loaded_at` (load time in UTC).
- The files in `data/raw` are only read, never modified.

## How the incremental load works

- `bronze.load_log` records every file that has been loaded: file name, target table, number of rows, and load time.
- On each run, the notebook lists the files in `data/raw` and loads only the files that are not in the log yet. Their rows are appended (`INSERT`); existing rows are never replaced.
- Running the notebook twice loads nothing the second time. This is called idempotent.
- New files are picked up by name pattern: a future `transactions_2020-10.csv` would be appended to `bronze.transactions`.
- Loading a file and writing its log entry happen in one database transaction. If loading fails halfway, both are rolled back, so a file is never half loaded or loaded twice.

Why the log works per file and not per row: identical transaction rows are real purchases (the dataset has no quantity column), so duplicates cannot be detected per row. Tracking files prevents the same delivery from being loaded twice.

Limitation: a file is identified by its name. A file that is changed but keeps the same name is not reloaded; use `FULL_REFRESH` for that.

Before running: select the `.venv` kernel and close any other program that has the database open (DuckDB allows only one writer at a time).

## Reading guide

Follow the notebook from top to bottom: find the source files, prepare the load log, load new files, check the result, and inspect a few rows. Run all cells in order; the database connection closes in the final cell.

## 1. Settings and source files

Keep `FULL_REFRESH = False` for normal incremental runs. The setup code finds the expected CSV files in `data/raw` and stops if a source table has no matching file. Its printed file list shows what Bronze can load.

In [ ]:
# True rebuilds bronze from scratch: all bronze tables and the load log are dropped and every file is loaded again.
FULL_REFRESH = False

In [ ]:
from pathlib import Path

import duckdb

# Works whether the notebook runs from the project root or from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "crossselliq.duckdb"

# Bronze table name -> file name pattern in data/raw.
SOURCE_PATTERNS = {
    "articles": "articles*.csv",
    "customers": "customers*.csv",
    "transactions": "transactions*.csv",
}
METADATA_COLUMNS = ["_source_file", "_loaded_at"]

source_files = [
    (table_name, path)
    for table_name, pattern in SOURCE_PATTERNS.items()
    for path in sorted(RAW_DIR.glob(pattern))
]

missing_tables = [t for t in SOURCE_PATTERNS if t not in {table for table, _ in source_files}]
if missing_tables:
    raise FileNotFoundError(f"No raw file found for: {missing_tables} in {RAW_DIR}")

print(f"Raw folder: {RAW_DIR}")
print(f"Database:   {DB_PATH}")
for table_name, path in source_files:
    print(f"  found {path.name} -> bronze.{table_name}")

## 2. Prepare the Bronze schema and load log

If the load log does not exist yet while bronze tables do (for example from the earlier full-reload version of this notebook), the notebook cannot know which files those tables contain. Appending would then load everything twice, so bronze is rebuilt from scratch in that case.

In [ ]:
con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

load_log_exists = con.execute(
    "SELECT count(*) FROM information_schema.tables WHERE table_schema = 'bronze' AND table_name = 'load_log'"
).fetchone()[0] == 1

if FULL_REFRESH or not load_log_exists:
    reason = "FULL_REFRESH is True" if FULL_REFRESH else "no load log found"
    print(f"Rebuilding bronze from scratch ({reason}).")
    # Table names come from the fixed SOURCE_PATTERNS mapping, not from user input.
    for table_name in [*SOURCE_PATTERNS, "load_log"]:
        con.execute(f"DROP TABLE IF EXISTS bronze.{table_name}")

con.execute(
    """
    CREATE TABLE IF NOT EXISTS bronze.load_log (
        file_name  VARCHAR PRIMARY KEY,
        table_name VARCHAR NOT NULL,
        row_count  BIGINT NOT NULL,
        loaded_at  TIMESTAMP NOT NULL
    )
    """
)

already_loaded = {row[0] for row in con.execute("SELECT file_name FROM bronze.load_log").fetchall()}
new_files = [(table_name, path) for table_name, path in source_files if path.name not in already_loaded]

print(f"Files already loaded: {len(already_loaded)}")
print(f"New files to load:    {len(new_files)}")

## 3. Load new files

`all_varchar = true` stops DuckDB from guessing types. That keeps the leading zero in `article_id` and makes sure no value is changed or rejected during loading.

`INSERT ... BY NAME` matches columns by name instead of position. If a new file has a column the table does not know, the insert fails instead of silently putting values in the wrong column.

The first run loads the 3.5 GB transactions file, so it takes a while. Later runs without new files finish immediately.

In [ ]:
import time

# current_timestamp is fixed for the duration of a transaction, so the rows and their
# load log entry get exactly the same _loaded_at value.
SELECT_FILE_SQL = """
    SELECT
        *,
        $file_name AS _source_file,
        current_timestamp AT TIME ZONE 'UTC' AS _loaded_at
    FROM read_csv($path, header = true, all_varchar = true)
"""


def load_file(con, table_name, path):
    """Append one raw file to its bronze table and record it in the load log, atomically."""
    params = {"path": str(path), "file_name": path.name}
    con.begin()
    try:
        # The table name comes from the fixed SOURCE_PATTERNS mapping, and identifiers
        # cannot be SQL parameters. The file path and name are passed as parameters.
        con.execute(f"CREATE TABLE IF NOT EXISTS bronze.{table_name} AS {SELECT_FILE_SQL} LIMIT 0", params)
        row_count = con.execute(f"INSERT INTO bronze.{table_name} BY NAME {SELECT_FILE_SQL}", params).fetchone()[0]
        con.execute(
            """
            INSERT INTO bronze.load_log (file_name, table_name, row_count, loaded_at)
            VALUES ($file_name, $table_name, $row_count, current_timestamp AT TIME ZONE 'UTC')
            """,
            {"file_name": path.name, "table_name": table_name, "row_count": row_count},
        )
        con.commit()
    except Exception:
        con.rollback()
        raise
    return row_count


if not new_files:
    print("No new files: bronze is up to date.")

for table_name, path in new_files:
    start = time.perf_counter()
    row_count = load_file(con, table_name, path)
    print(f"{path.name:<24} -> bronze.{table_name:<13} {row_count:>12,} rows in {time.perf_counter() - start:.1f} s")

## 4. Check the loaded data

Bronze is correct when nothing was lost, changed, or loaded twice:

1. Every file in `data/raw` is in the load log.
2. Per file: the rows in bronze equal the rows in the load log and the rows in the CSV file.
3. Every bronze row belongs to a file in the load log.
4. Same column names as the CSV, plus the two metadata columns, and all source columns stored as `VARCHAR`.

The last cell stops with an error if any check fails.

In [ ]:
results = []


def check(name, passed):
    results.append((name, passed))
    print(f"  {'OK  ' if passed else 'FAIL'} {name}")


def scalar(sql, params=None):
    return con.execute(sql, params or {}).fetchone()[0]


logged = {row[0]: row[1] for row in con.execute("SELECT file_name, row_count FROM bronze.load_log").fetchall()}

print("load log")
check("every raw file is in the load log", all(path.name in logged for _, path in source_files))

for table_name, path in source_files:
    raw_rows = scalar("SELECT count(*) FROM read_csv($path, header = true, all_varchar = true)", {"path": str(path)})
    bronze_rows = scalar(f"SELECT count(*) FROM bronze.{table_name} WHERE _source_file = $file_name", {"file_name": path.name})
    print(f"{path.name}: {bronze_rows:,} rows in bronze.{table_name}")
    check("rows match the CSV file", bronze_rows == raw_rows)
    check("rows match the load log", bronze_rows == logged.get(path.name))

for table_name in SOURCE_PATTERNS:
    print(f"bronze.{table_name}")
    orphan_rows = scalar(
        f"SELECT count(*) FROM bronze.{table_name} WHERE _source_file NOT IN (SELECT file_name FROM bronze.load_log)"
    )
    check("every row belongs to a logged file", orphan_rows == 0)

    first_file = next(path for t, path in source_files if t == table_name)
    raw_columns = [
        row[0]
        for row in con.execute(
            "DESCRIBE SELECT * FROM read_csv($path, header = true, all_varchar = true)", {"path": str(first_file)}
        ).fetchall()
    ]
    bronze_schema = con.execute(f"DESCRIBE bronze.{table_name}").fetchall()
    check("columns match the CSV + metadata", [row[0] for row in bronze_schema] == raw_columns + METADATA_COLUMNS)
    check("all source columns are VARCHAR",
          all(row[1] == "VARCHAR" for row in bronze_schema if row[0] not in METADATA_COLUMNS))

In [ ]:
failed = [name for name, passed in results if not passed]
if failed:
    raise AssertionError(f"{len(failed)} bronze check(s) failed: {failed}")
print(f"All {len(results)} bronze checks passed.")

## 5. Inspect the result

The load log shows which files were loaded and when. The small row previews help confirm that source values and the two metadata columns are visible without printing the full tables.

In [ ]:
con.sql("SELECT file_name, table_name, row_count, loaded_at FROM bronze.load_log ORDER BY loaded_at, file_name").show()

for table_name in SOURCE_PATTERNS:
    print(f"bronze.{table_name}")
    con.sql(f"SELECT * FROM bronze.{table_name} LIMIT 3").show()

## 6. Close the database

Always close the connection, otherwise the database file stays locked for other programs.

In [ ]:
con.close()